# Notebook 05 (Colab) — Avaliação Framework TILD

**Objetivo:** Avaliar todos os modelos treinados usando o framework TILD:

| Dimensão | Métrica | Descrição |
|---|---|---|
| T (Técnica) | P/R/F1 | seqeval entity-level, esquema BIO |
| I (Informacional) | ΔF1 | F1_anonimizado − F1_original |
| L (Legal) | Coverage/Prec_anon | % de PHI anonimizado |

**Referência:** Schiezaro et al. (2026)

---

In [ ]:
import sys, os
from pathlib import Path

REPO_DIR = "/content/anonimizacao_clinica"
DRIVE_PROJECT = "/content/drive/MyDrive/mestrado_anonimizacao"

sys.path.insert(0, REPO_DIR)

from src.evaluation.metrics import NERMetrics, AnonymizationMetrics
from src.ner.labels import ALL_LABELS, bio_to_spans
from src.ner.models import ALL_MODEL_NAMES

print("Modelos a avaliar:", ALL_MODEL_NAMES)

In [ ]:
# ── Carregar predições salvas ──────────────────────────────────────────────────
# Cada modelo deve ter salvado:
#   outputs/results/{model_name}/predictions.jsonl
#   Formato: {"tokens": [...], "y_true": [...], "y_pred": [...]}

RESULTS_DIR = Path(DRIVE_PROJECT) / "outputs" / "results"

available_models = [
    d.name for d in RESULTS_DIR.iterdir() 
    if d.is_dir() and (d / "predictions.jsonl").exists()
] if RESULTS_DIR.exists() else []

print(f"Modelos com predições disponíveis: {available_models}")

In [ ]:
# ── Calcular métricas T (seqeval) para todos os modelos ───────────────────────
import json
import pandas as pd

rows = []
for model_name in available_models:
    pred_file = RESULTS_DIR / model_name / "predictions.jsonl"
    y_true, y_pred = [], []
    
    with open(pred_file) as f:
        for line in f:
            record = json.loads(line)
            y_true.append(record["y_true"])
            y_pred.append(record["y_pred"])
    
    metrics = NERMetrics.compute_seqeval(y_true, y_pred)
    rows.append({
        "Modelo": model_name,
        "Precision": metrics["overall_precision"],
        "Recall":    metrics["overall_recall"],
        "F1":        metrics["overall_f1"],
    })

if rows:
    df_results = pd.DataFrame(rows).sort_values("F1", ascending=False)
    display(df_results.style.format({"Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"}))
else:
    print("Nenhuma predição disponível ainda.")

In [ ]:
# ── Cálculo de ΔF1 (dimensão I — Informacional) ───────────────────────────────
#
# Protocolo:
# 1. Treinar modelo downstream NER de utilidade (MEDICAMENTO, DOSE, etc.)
#    no corpus ORIGINAL
# 2. Aplicar anonimização PHI → texto_anonimizado
# 3. Avaliar modelo downstream no corpus ANONIMIZADO
# 4. ΔF1 = F1_anon − F1_orig  (quanto de utilidade clínica foi preservado)
#
# TODO: implementar após treinamento dos modelos

print("ΔF1: implementar após treinamento dos modelos de utilidade clínica.")
print("Ver src/evaluation/metrics.py → AnonymizationMetrics.compute_delta_f1()")

In [ ]:
# ── Exportar Tabela de Resultados para dissertação ────────────────────────────
if 'df_results' in dir() and len(df_results) > 0:
    output_path = f"{DRIVE_PROJECT}/outputs/results/tabela_resultados.xlsx"
    df_results.to_excel(output_path, index=False)
    print(f"✓ Tabela exportada: {output_path}")